# OpenAI Inference

## Imports

In [ ]:
import sys
import json
from pathlib import Path
import pandas as pd

sys.path.append('../..')

from utils.llm_evaluation_utils import *
from utils.prompt_builder import build_rating_prompt, build_critique_prompt
from utils.data_setup import get_dataset_file_path, prepare_project_data
from utils.models_setup import setup_client, query_openai_model
from utils.openai_batch_manager import OpenAIBatchManager


## Config and Constants

In [ ]:
TASK_SUBSET = "all_tasks"       # "all_tasks" | "1000_tasks" | "50_tasks"
PROMPTING_TYPE = "zero_shot"    # "zero_shot" | "few_shot"
STAGE = "critiques"             # "ratings" | "critiques"
NUM_TRIALS = 1
MAX_TOKENS = 30000
TEMPERATURE = 1
PROVIDER = "openai"             # "openai" | "qwen" | "deepseek"

if STAGE == "ratings":
    SYSTEM_MESSAGE = RATING_SYSTEM_MESSAGE
    build_prompt_fn = build_rating_prompt
    update_results_df_fn = update_rating_in_df
elif STAGE == "critiques":
    SYSTEM_MESSAGE = CRITIQUES_SYSTEM_MESSAGE
    build_prompt_fn = build_critique_prompt
    update_results_df_fn = update_critiques_in_df
else:
    raise ValueError(f"Invalid stage: {STAGE}")

client, cfg = setup_client(
    provider=PROVIDER,
    system_message=SYSTEM_MESSAGE,
    temperature=TEMPERATURE,
    max_tokens=MAX_TOKENS,
)

cfg


## Data Loading

In [ ]:
uicrit_file, base64_screens_file, few_shot_samples_file = get_dataset_file_path(STAGE)

uicrit_df = pd.read_parquet(uicrit_file)
base64_screens_df = pd.read_parquet(base64_screens_file)

if few_shot_samples_file:
    few_shot_samples_df = pd.read_parquet(few_shot_samples_file)
else:
    few_shot_samples_df = None

print("UICrit rows:", len(uicrit_df))
print("Screens rows:", len(base64_screens_df))

In [ ]:
schedule_dir = Path("./schedules")
schedule_dir.mkdir(exist_ok=True)

rating_schedule_file_1000_tasks = schedule_dir / "rating_trial_schedule-1000_tasks.parquet"
rating_schedule_file_all_tasks = schedule_dir / "rating_trial_schedule-all_tasks.parquet"
critiques_schedule_file_1000_tasks = schedule_dir / "critiques_trial_schedule-1000_tasks.parquet"
critiques_schedule_file_all_tasks = schedule_dir / "critiques_trial_schedule-all_tasks.parquet"
critiques_schedule_file_all_tasks_excluding_1000 = schedule_dir / "critiques_trial_schedule-all_tasks_excluding_1000.parquet"


## Preparing Schedules

Use these cells only when you need to build or refresh the schedule parquet files under `./schedules`.

### Ratings Schedule

In [ ]:
# Build ratings schedule for all tasks
records = []
for _, row in uicrit_df.iterrows():
    screen_task_id = row["screen_task_id"]
    for trial in range(1, NUM_TRIALS + 1):
        for aspect in EVALUATION_MAIN_ASPECTS:
            record = {
                "custom_id": (
                    f"{screen_task_id}&trial{trial}&aspect={aspect}"
                    if NUM_TRIALS > 1
                    else f"{screen_task_id}&aspect={aspect}"
                ),
                "screen_task_id": screen_task_id,
                "aspect": aspect,
            }
            if NUM_TRIALS > 1:
                record["trial"] = trial
            records.append(record)

trials_schedule_df = pd.DataFrame.from_records(records)
trials_schedule_df.to_parquet(rating_schedule_file_all_tasks, index=False)

print(trials_schedule_df.shape)
trials_schedule_df.head(4)

In [ ]:
# Build ratings schedule for the first task of 1000 screens
trials_schedule_df = pd.read_parquet(rating_schedule_file_all_tasks)
first_tasks_df = uicrit_df.drop_duplicates(subset="screen_id", keep="first")
selected_task_ids = first_tasks_df["screen_task_id"].head(1000).unique()

trials_schedule_df = trials_schedule_df[trials_schedule_df["screen_task_id"].isin(selected_task_ids)].copy()
trials_schedule_df.to_parquet(rating_schedule_file_1000_tasks, index=False)

print(trials_schedule_df.shape)
trials_schedule_df.head(4)

In [ ]:
# Optional: rebuild few-shot samples file for ratings
uicrit_df_ratings = uicrit_df.copy()
uicrit_df_ratings["total_rating"] = uicrit_df_ratings[EVALUATION_FIVE_ASPECTS].sum(axis=1)
uicrit_df_ratings = uicrit_df_ratings.sort_values(by="total_rating")

lowest_rated = uicrit_df_ratings.head(3).drop(columns=["total_rating", "comments_source", "comments"], errors="ignore")
mid_rated = uicrit_df_ratings.iloc[[len(uicrit_df_ratings) // 2]].drop(columns=["total_rating", "comments_source", "comments"], errors="ignore")
highest_rated = uicrit_df_ratings.tail(3).drop(columns=["total_rating", "comments_source", "comments"], errors="ignore")

few_shot_samples_df = pd.concat([lowest_rated, mid_rated, highest_rated]).merge(base64_screens_df, on="screen_id", how="left")
few_shot_samples_df.to_parquet(few_shot_samples_file, index=False)

few_shot_samples_df.head()

### Critiques Schedule

In [ ]:
# Build critiques schedule for all tasks
records = []
for _, row in uicrit_df.iterrows():
    screen_task_id = row["screen_task_id"]
    records.append({
        "custom_id": screen_task_id,
        "screen_task_id": screen_task_id,
    })

critiques_schedule_df = pd.DataFrame.from_records(records)
critiques_schedule_df.to_parquet(critiques_schedule_file_all_tasks, index=False)

print(critiques_schedule_df.shape)
critiques_schedule_df.head(4)

In [ ]:
# Build critiques schedule for the first task of 1000 screens
critiques_schedule_df = pd.read_parquet(critiques_schedule_file_all_tasks)
first_tasks_df = uicrit_df.drop_duplicates(subset="screen_id", keep="first")
selected_task_ids = first_tasks_df["screen_task_id"].head(1000).unique()

critiques_schedule_df = critiques_schedule_df[critiques_schedule_df["screen_task_id"].isin(selected_task_ids)].copy()
critiques_schedule_df.to_parquet(critiques_schedule_file_1000_tasks, index=False)

print(critiques_schedule_df.shape)
critiques_schedule_df.head(4)

In [ ]:
# Optional: build critiques schedule for all_tasks excluding the 1000-task subset
schedule_1000_tasks_df = pd.read_parquet(critiques_schedule_file_1000_tasks)
all_tasks_schedule_df = pd.read_parquet(critiques_schedule_file_all_tasks)

schedule_all_tasks_excluding_1000_df = all_tasks_schedule_df[
    ~all_tasks_schedule_df["screen_task_id"].isin(schedule_1000_tasks_df["screen_task_id"])
].copy()

schedule_all_tasks_excluding_1000_df.to_parquet(critiques_schedule_file_all_tasks_excluding_1000, index=False)

print(schedule_all_tasks_excluding_1000_df.shape)
schedule_all_tasks_excluding_1000_df.head(4)

## Select Schedule For The Current Run

In [ ]:
if TASK_SUBSET == "1000_tasks":
    if STAGE == "ratings":
        schedule_file = rating_schedule_file_1000_tasks
    else:
        schedule_file = critiques_schedule_file_1000_tasks
else:
    if STAGE == "ratings":
        schedule_file = rating_schedule_file_all_tasks
    else:
        schedule_file = critiques_schedule_file_all_tasks

print("Using schedule file:", schedule_file)


## Load Responses DataFrame

In [ ]:
responses_df, results_paths = prepare_project_data(
    model_short=cfg["model_short"],
    num_trials=NUM_TRIALS,
    task_subset=TASK_SUBSET,
    shots=PROMPTING_TYPE,
    stage=STAGE,
)

text_responses_jsonl_file = results_paths["results_jsonl"]
model_results_file = results_paths["results_parquet"]

print("Responses shape:", responses_df.shape)
responses_df.head(2)

## Batch Setup

In [ ]:
batch_main_dir = Path(f"./{STAGE}_Batching/Batch_Files-{TASK_SUBSET}-{PROMPTING_TYPE}-{PROVIDER}")

batch_process = OpenAIBatchManager(
    client=client,
    batch_main_dir=batch_main_dir,
    selected_tasks=TASK_SUBSET,
)

batch_main_dir

## Prepare Prompts JSONL

In [ ]:
prompts_file = batch_process.generate_prompts_jsonl(
    schedule_file=schedule_file,
    responses_df=responses_df,
    screens_df=base64_screens_df,
    model_version=cfg["model_version"],
    system_message=SYSTEM_MESSAGE,
    guidelines=GUIDELINES,
    temperature=TEMPERATURE,
    max_tokens=MAX_TOKENS,
    prompting_type=PROMPTING_TYPE,
    samples_df=few_shot_samples_df,
    image_format="jpeg",
    build_prompt_fn=build_prompt_fn,
)

prompts_file

In [ ]:
# Preview the first few JSONL lines
with open(prompts_file, "r", encoding="utf-8") as f:
    for _ in range(4):
        print(json.loads(f.readline()))

In [ ]:
# Split prompts into smaller batch files as needed
batch_process.split_prompts_jsonl(max_lines_per_file=1000)

## Batch Requests

Use the manual section when you want fine-grained control. Use the automatic section when you want the notebook to upload, create, poll, and download sequentially.

### Manual Batch Workflow

In [ ]:
# Upload one split prompt file manually
batch_file_path = batch_process.prompts_dir / "prompts-batch_1.jsonl"
input_file_id = batch_process.upload_file(batch_file_path)
input_file_id

In [ ]:
# List uploaded files
_ = batch_process.list_uploaded_files(purpose="batch")

In [ ]:
# Create a batch manually for a previously uploaded file
# input_file_id = "file-XXXXXXXXXXXXXXXXXXXXXXXX"
batch = batch_process.create_batch(input_file_id)
batch

In [ ]:
# List recent batches
_ = batch_process.list_batches(limit=100)

In [ ]:
# Check a batch and retrieve its status
# batch_id = "batch_XXXXXXXXXXXXXXXXXXXXXXXX"
state, batch = batch_process.check_batch(
    verbose=True,
    # batch_id=batch_id,
)
state

In [ ]:
# Retrieve batch output after completion
responses_file_path = batch_process.retrieve_batch_output(batch_obj=batch, base_name="responses")
responses_file_path

### Automatic Batch Workflow

In [ ]:
start_from_batch = 1
resume_batch = None

batch_process.process_batches_from(
    start_from_batch=start_from_batch,
    sleep_minutes=2,
    resume_batch_id=resume_batch,
    base_name="responses",
)

## Extract Results From Batch Responses

In [ ]:
summary = batch_process.extract_results_from_responses(
    responses_df,
    update_results_df_fn,
)

print(
    "Updated:", summary["updated"],
    "| Skipped:", len(summary["skipped"]),
    "| Errors:", len(summary["errors"]),
)

responses_df.head(2)

In [ ]:
# Optional: combine all response files into one JSONL for inspection
from pathlib import Path

responses_dir = Path(batch_process.responses_dir)
output_file = Path(batch_process.prompts_file).parent / f"responses-{TASK_SUBSET}.jsonl"

count = 0
with output_file.open("w", encoding="utf-8") as out_f:
    for p in sorted(responses_dir.glob("responses*.jsonl")):
        if not p.is_file():
            continue
        if p.resolve() == output_file.resolve():
            continue

        text = p.read_text(encoding="utf-8", errors="ignore")
        if text:
            if not text.endswith("\n"):
                text += "\n"
            out_f.write(text)
            count += 1

print(f"Written {count} files to {output_file}")


## Alternative: One-By-One Requests

Use this section instead of batching when you want to test prompts or run a very small sample interactively.

In [ ]:
responses_df_sample = responses_df.head(1).copy()
responses_df_sample

In [ ]:
query_args = dict(
    client=client,
    model_version=cfg["model_version"],
    max_tokens=MAX_TOKENS,
    temperature=TEMPERATURE,
    system=SYSTEM_MESSAGE,
    image_format="jpeg",
)

run_llm_inference(
    responses_df=responses_df_sample,
    base64_screens_df=base64_screens_df,
    few_shot_samples_df=few_shot_samples_df,
    evaluation_aspects=EVALUATION_MAIN_ASPECTS,
    build_prompt_fn=build_prompt_fn,
    update_results_df_fn=update_results_df_fn,
    query_fn=query_openai_model,
    query_args=query_args,
    save_jsonl_fn=save_response_text,
    output_jsonl=text_responses_jsonl_file,
    guidelines=GUIDELINES,
    prompting_type=PROMPTING_TYPE,
    requests_per_minute=cfg["rpm"],
    stage=STAGE,
    print_output=True,
)

## Explore Results

In [ ]:
responses_df.tail(5)

In [ ]:
if STAGE == "critiques":
    columns_with_none = (responses_df.isna() | (responses_df == "")).sum()
else:
    columns_with_none = responses_df[EVALUATION_FIVE_ASPECTS].isna().sum()

columns_with_none

In [ ]:
if STAGE == "critiques":
    rows_with_none = responses_df[responses_df["critiques"].isna() | (responses_df["critiques"] == "")]
else:
    rows_with_none = responses_df[responses_df[EVALUATION_FIVE_ASPECTS].isna().any(axis=1)]

rows_with_none

In [ ]:
if STAGE == "critiques":
    missing = (responses_df["critiques"].isna() | (responses_df["critiques"] == "")).sum()
else:
    missing = responses_df[EVALUATION_FIVE_ASPECTS].isna().any(axis=1).sum()

print("rows:", len(responses_df))
print("unique screen_task_id:", responses_df["screen_task_id"].nunique())
print("missing outputs:", missing)

## Save Results

In [ ]:
responses_df.to_parquet(model_results_file, index=False)
print("Saved:", model_results_file)